In [1]:
import os
from google.colab import drive
import zipfile

drive.mount('/content/drive')

ZIP_PATH = '/content/drive/MyDrive/Image Segmentation Project/Project/Task 8/Anomaly_Validation_Datasets.zip'
EXTRACT_DIR = '/content/datasets'

if not os.path.exists(EXTRACT_DIR):
    os.makedirs(EXTRACT_DIR)
    print("Unzipping datasets... this might take a minute.")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print("Unzip complete!")
else:
    print("Datasets already extracted.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Datasets already extracted.


In [2]:
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import auc, roc_curve, precision_recall_curve

def calculate_anomaly_metrics(y_true, y_scores, ignore_index=255):
    """
    Calculates AuPRC and FPR95 for anomaly segmentation.
    y_true: Ground truth numpy array (0 = Normal, 1 = Anomaly, 255 = Ignore)
    y_scores: Anomaly score numpy array (Higher = more anomalous)
    """
    y_true_flat = y_true.flatten()
    y_scores_flat = y_scores.flatten()

    valid_mask = (y_true_flat != ignore_index)
    y_true_filtered = y_true_flat[valid_mask]
    y_scores_filtered = y_scores_flat[valid_mask]

    # Force binarization: 0 stays 0, anything else (like 2) becomes 1
    y_true_filtered = (y_true_filtered > 0).astype(int)

    if len(y_true_filtered) == 0 or len(np.unique(y_true_filtered)) < 2:
        return 0.0, 0.0

    precision, recall, _ = precision_recall_curve(y_true_filtered, y_scores_filtered)
    auprc = auc(recall, precision)

    fpr, tpr, thresholds = roc_curve(y_true_filtered, y_scores_filtered)
    idx = np.where(tpr >= 0.95)[0][0]

    return auprc, fpr[idx]

In [3]:
# Elles prennent des logits pixel-level [B, C, H, W] en entrée,
# peu importe que ce soit ERFNet ou EoMT qui les ait produits.

def get_msp_score(logits):
    """ MSP Score: 1 - max(softmax(logits)) """
    probs = F.softmax(logits, dim=1)
    max_probs, _ = torch.max(probs, dim=1)
    anomaly_score = 1.0 - max_probs
    return anomaly_score.cpu().numpy()

def get_maxlogit_score(logits):
    """ MaxLogit Score: -max(logits) """
    max_logits, _ = torch.max(logits, dim=1)
    anomaly_score = -max_logits
    return anomaly_score.cpu().numpy()

def get_max_entropy_score(logits):
    """ Max Entropy Score: H = -sum(p * log(p)), normalisée par log(C) """
    probs = F.softmax(logits, dim=1)
    raw_entropy = torch.sum(-probs * torch.log(probs + 1e-8), dim=1)
    num_classes = probs.shape[1]
    normalized_entropy = torch.div(raw_entropy, torch.log(torch.tensor(num_classes, dtype=torch.float32, device=logits.device)))
    return normalized_entropy.cpu().numpy()


#
# EoMT (architecture à masques) retourne :
#   - pred_masks  : [B, N, H, W]  — N masques binaires (un par query)
#   - pred_logits : [B, N, C+1]   — logits de classe par query (+1 = classe "vide")
#
# On reconstruit [B, C, H, W] par combinaison linéaire :
#   pixel_logits[b, c, h, w] = sum_n( softmax(class_logits)[b,n,c] * sigmoid(mask)[b,n,h,w] )

def convert_eomt_output_to_pixel_logits(outputs):
  """Convertit la sortie EoMT en logits pixel-level [B, C, H, W].
    Returns:
        pixel_logits : Tensor [B, C, H, W] utilisable par get_msp_score, etc.
    """
  # outputs[0][-1] : masques du dernier bloc  [B, 100, H, W]
  # outputs[1][-1] : logits du dernier bloc   [B, 100, 20]
  mask_logits  = outputs[0][-1]  # [B, N, H, W]
  class_logits = outputs[1][-1]  # [B, N, C]  — pas de classe "no-object" ici (20 = 19+1 void)

  # Retirer la dernière classe void (index -1)
  class_logits_no_void = class_logits[:, :, :-1]  # [B, N, 19]

  class_probs = F.softmax(class_logits_no_void, dim=-1)  # [B, N, 19]
  masks = torch.sigmoid(mask_logits)                     # [B, N, H, W]

  pixel_logits = torch.einsum('bnc,bnhw->bchw', class_probs, masks)  # [B, 19, H, W]
  return pixel_logits


#
# Elle ne passe pas par les logits pixel-level mais exploite directement
# les masques bruts de chaque query.
#
# Idée : un pixel est anomal si AUCUNE query ne le revendique avec confiance.
# Score RbA par pixel = 1 - max_n( sigmoid(mask[n]) * max_class_prob[n] )
# Référence : "RbA: Segmenting Unknown Regions Rejected by All" (ref [7] du projet)


def get_rba_score(outputs):
    #RbA Score : 1 - max_query( sigmoid(mask) * max_class_prob )
    mask_logits  = outputs[0][-1]  # [B, N, H, W]
    class_logits = outputs[1][-1]  # [B, N, C]

    # Retirer la classe "no-object" avant softmax
    class_logits_no_void = class_logits[:, :, :-1]  # [B, N, 19]
    class_probs = F.softmax(class_logits_no_void, dim=-1)  # [B, N, 19]
    max_class_prob, _ = class_probs.max(dim=-1)  # [B, N] # Probabilité max de classe pour chaque query : "à quel point cette query est-elle sûre ?"

    masks = torch.sigmoid(mask_logits)  # [B, N, H, W], proba spatiale de chaque mask
    claim = masks * max_class_prob.unsqueeze(-1).unsqueeze(-1)  # [B, N, H, W] # Score de revendication par pixel et par query
    # On broadcast max_class_prob [B, N] -> [B, N, H, W]
    # Max sur toutes les queries : le score de la query la plus confiante
    max_claim, _ = claim.max(dim=1)  # [B, H, W]

    return (1.0 - max_claim).cpu().numpy()

In [4]:
import sys, os
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import auc, roc_curve, precision_recall_curve

SHARED_FOLDER_NAME = 'Image Segmentation Project/Project/Task 8'

EOMT_REPO_PATH = '/content/drive/MyDrive/Image Segmentation Project/Project/MaskArchitectureAnomaly_CourseProject/eomt'
sys.path.append(EOMT_REPO_PATH)

from models.eomt import EoMT  # adapter selon la structure du repo fourni
from models.vit import ViT

# Vous devez évaluer 3 checkpoints :
#   - 'eomt_coco.pth'        : entraîné sur COCO (fourni)
#   - 'eomt_cityscapes.pth'  : entraîné sur Cityscapes (fourni)
#   - 'eomt_finetuned.pth'   : votre fine-tune (Task 5)
# Changer cette variable et relancer la Cell 5 + Cell 6 pour chaque checkpoint.
CHECKPOINT_NAME = 'eomt_cityscapes.bin'


# Le dataset ne dépend pas du modèle.
def calculate_anomaly_metrics(y_true, y_scores, ignore_index=255):
    y_true_flat = y_true.flatten()
    y_scores_flat = y_scores.flatten()
    valid_mask = (y_true_flat != ignore_index)
    y_true_filtered = y_true_flat[valid_mask]
    y_scores_filtered = y_scores_flat[valid_mask]
    y_true_filtered = (y_true_filtered > 0).astype(int)
    if len(y_true_filtered) == 0 or len(np.unique(y_true_filtered)) < 2:
        return 0.0, 0.0
    precision, recall, _ = precision_recall_curve(y_true_filtered, y_scores_filtered)
    auprc = auc(recall, precision)
    fpr, tpr, _ = roc_curve(y_true_filtered, y_scores_filtered)
    idx = np.where(tpr >= 0.95)[0][0]
    return auprc, fpr[idx]

class CustomAnomalyDataset(Dataset):
    def __init__(self, root, input_transform=None, target_transform=None):
        self.images_dir = os.path.join(root, 'images')
        self.masks_dir = os.path.join(root, 'labels_masks')
        image_files = sorted([f for f in os.listdir(self.images_dir) if not f.startswith('.')])
        self.samples = []
        for img_file in image_files:
            stem = os.path.splitext(img_file)[0]
            mask_files = [m for m in os.listdir(self.masks_dir) if m.startswith(stem + '.')]
            if mask_files:
                self.samples.append((img_file, mask_files[0]))
        self.input_transform = input_transform
        self.target_transform = target_transform

    def __getitem__(self, index):
        img_name, mask_name = self.samples[index]
        img_path = os.path.join(self.images_dir, img_name)
        mask_path = os.path.join(self.masks_dir, mask_name)
        with open(img_path, 'rb') as f: image = Image.open(f).convert('RGB')
        with open(mask_path, 'rb') as f: mask = Image.open(f).convert('P')
        if self.input_transform: image = self.input_transform(image)
        if self.target_transform: mask = self.target_transform(mask)
        return image, mask, img_name, mask_name

    def __len__(self):
        return len(self.samples)

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = 1

#
#   model = ERFNet(20).to(device)
#   checkpoint = torch.load(weights_path, map_location=device)
#   cleaned_state_dict = {k.replace('module.', ''): v for k, v in checkpoint.items()}
#   model.load_state_dict(cleaned_state_dict, strict=False)
#
# Les 3 checkpoints ont des configurations différentes :
#   - eomt_coco.bin       : img_size=640, num_q=100, num_classes=133
#   - eomt_cityscapes.bin : img_size=896, num_q=100, num_classes=19
#   - eomt_finetuned.bin  : img_size=640, num_q=200, num_classes=19
print(f"Loading EoMT with checkpoint: {CHECKPOINT_NAME}...")

if CHECKPOINT_NAME == 'eomt_cityscapes.bin':
    img_size    = 1024
    num_q       = 100
    num_classes = 19
elif CHECKPOINT_NAME == 'eomt_finetuned.bin':
    img_size    = 640
    num_q       = 200
    num_classes = 19
else:  # eomt_coco.bin
    img_size    = 640
    num_q       = 100
    num_classes = 133

encoder = ViT(
    backbone_name='vit_base_patch14_reg4_dinov2',
    img_size=(img_size, img_size),
    patch_size=16 if CHECKPOINT_NAME == 'eomt_cityscapes.bin' else 14,
    ckpt_path=None
)

model = EoMT(
    encoder=encoder,
    num_classes=num_classes,  # 133 pour COCO, 19 pour Cityscapes et finetuned
    num_q=num_q,              # 100 pour COCO et Cityscapes, 200 pour finetuned
    num_blocks=3,             # valeur du YAML — identique pour tous les checkpoints
    masked_attn_enabled=True
).to(device)

weights_path = f'/content/drive/MyDrive/{SHARED_FOLDER_NAME}/{CHECKPOINT_NAME}'
checkpoint = torch.load(weights_path, map_location=device)

if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
    state_dict = checkpoint['state_dict']
else:
    state_dict = checkpoint

# Retirer le préfixe 'network.' dans tous les cas
state_dict = {k.replace('network.', ''): v for k, v in state_dict.items()}

missing, unexpected = model.load_state_dict(state_dict, strict=False)
print(f"Missing keys  : {len(missing)} → {missing[:5] if missing else 'aucun'}")
print(f"Unexpected keys: {len(unexpected)} → {unexpected[:5] if unexpected else 'aucun'}")

model.eval()
print("Weights loaded successfully!")

#
#   -> ERFNet n'avait pas besoin de normalisation particulière.
#
#   -> EoMT utilise un backbone ViT (DINOv2) pré-entraîné sur ImageNet.
#      Sans cette normalisation, les activations seraient hors distribution.
#      Le resize doit correspondre à img_size du checkpoint chargé.
input_transform_fn = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # mean ImageNet
        std=[0.229, 0.224, 0.225]    # std ImageNet
    )
])


Loading EoMT with checkpoint: eomt_cityscapes.bin...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Missing keys  : 0 → aucun
Unexpected keys: 1 → ['criterion.empty_weight']
Weights loaded successfully!


In [6]:
def evaluate_single_dataset(dataset_name, base_dir='/content/datasets/Validation_Dataset'):
    print(f"\n--- Evaluating on {dataset_name} (checkpoint: {CHECKPOINT_NAME}) ---")

    val_dataset = CustomAnomalyDataset(
        root=f'{base_dir}/{dataset_name}',
        input_transform=input_transform_fn,
        target_transform=target_transform_fn
    )
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    all_gt, all_msp, all_maxlogit, all_entropy, all_rba = [], [], [], [], []

    for images, masks, _, _ in tqdm(val_loader, desc=f"Processing"):
        images = images.to(device)
        masks = masks.squeeze(1).numpy()

        # Les grandes images (ex: RoadObsticle21) saturaient la RAM
        # car les scores étaient stockés en haute résolution.
        # On force tout à img_size x img_size — GT, logits et RbA.
        import cv2
        masks = np.stack([
            cv2.resize(m, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
            for m in masks
        ])
        h_orig, w_orig = img_size, img_size

        #
        #   logits = model(images)  # ERFNet retourne directement [B, C, H, W]
        #
        #   outputs = model(images)  # EoMT retourne un tuple (liste_masques, liste_logits)
        #   logits  = convert_eomt_output_to_pixel_logits(outputs)  # -> [B, C, H, W]
        #
        # On garde 'outputs' séparément car RbA en a besoin
        # (il travaille sur les masques bruts, pas sur les logits pixel-level).
        outputs = model(images)

        logits = convert_eomt_output_to_pixel_logits(outputs)  # [B, C, H, W]

        # EoMT prédit en résolution réduite — on upscale à img_size
        # qui est aussi la taille du GT redimensionné ci-dessus.
        logits = F.interpolate(logits, size=(h_orig, w_orig), mode='bilinear', align_corners=False)

        #
        # On ne passe plus par get_rba_score(outputs) car les masques EoMT
        # sont en résolution réduite — il faut les redimensionner avant le calcul
        # pour que rba_score ait la même taille que le ground truth.
        rba_masks_resized = F.interpolate(
            outputs[0][-1], size=(h_orig, w_orig), mode='bilinear', align_corners=False
        )  # [B, N, img_size, img_size]
        class_logits_no_void = outputs[1][-1][:, :, :-1]           # [B, N, C] — retirer classe void
        class_probs = F.softmax(class_logits_no_void, dim=-1)       # [B, N, C]
        max_class_prob, _ = class_probs.max(dim=-1)                 # [B, N]
        masks_sig = torch.sigmoid(rba_masks_resized)                # [B, N, img_size, img_size]
        claim = masks_sig * max_class_prob.unsqueeze(-1).unsqueeze(-1)  # [B, N, img_size, img_size]
        max_claim, _ = claim.max(dim=1)                             # [B, img_size, img_size]
        rba_score = (1.0 - max_claim).cpu().numpy()                 # anomal si aucune query ne revendique

        all_gt.append(masks.astype(np.uint8))
        all_msp.append(get_msp_score(logits).astype(np.float16))
        all_maxlogit.append(get_maxlogit_score(logits).astype(np.float16))
        all_entropy.append(get_max_entropy_score(logits).astype(np.float16))
        all_rba.append(rba_score.astype(np.float16))

        del images, outputs, logits, rba_masks_resized, masks_sig, claim, max_claim, rba_score
        torch.cuda.empty_cache()

    print(f"\nCalculating metrics for {dataset_name}...")

    gt_concat = np.concatenate(all_gt)
    del all_gt

    msp_concat = np.concatenate(all_msp)
    del all_msp
    msp_auprc, msp_fpr95 = calculate_anomaly_metrics(gt_concat, msp_concat)
    del msp_concat

    maxlogit_concat = np.concatenate(all_maxlogit)
    del all_maxlogit
    ml_auprc, ml_fpr95 = calculate_anomaly_metrics(gt_concat, maxlogit_concat)
    del maxlogit_concat

    entropy_concat = np.concatenate(all_entropy)
    del all_entropy
    ent_auprc, ent_fpr95 = calculate_anomaly_metrics(gt_concat, entropy_concat)
    del entropy_concat

    rba_concat = np.concatenate(all_rba)
    del all_rba
    rba_auprc, rba_fpr95 = calculate_anomaly_metrics(gt_concat, rba_concat)
    del rba_concat

    del gt_concat

    return {
        "MSP":         {"AuPRC": msp_auprc, "FPR95": msp_fpr95},
        "MaxLogit":    {"AuPRC": ml_auprc,  "FPR95": ml_fpr95},
        "Max Entropy": {"AuPRC": ent_auprc, "FPR95": ent_fpr95},
    }

In [7]:
import pandas as pd

if 'results_table' not in locals():
    results_table = {}

datasets_to_run = ["RoadAnomaly21", "RoadObsticle21", "FS_LostFound_full", "fs_static", "RoadAnomaly"]

with torch.no_grad():
    for d_name in datasets_to_run:
        # Cela permet de garder les résultats des 3 checkpoints
        # dans le même dictionnaire sans les écraser.
        results_table[f"{CHECKPOINT_NAME}_{d_name}"] = evaluate_single_dataset(d_name)

df_results = pd.DataFrame(results_table).round(4)

print(f"\nRisultati Finali — EoMT checkpoint: {CHECKPOINT_NAME}")
print(df_results.to_string(float_format='{:.3f}'.format))

csv_name = f'Task8_Results_{CHECKPOINT_NAME.replace(".bin", "")}.csv'
csv_path = f'/content/drive/MyDrive/{SHARED_FOLDER_NAME}/{csv_name}'
df_results.to_csv(csv_path)
print(f"\nTabella salvata in: {csv_path}")


--- Evaluating on RoadAnomaly21 (checkpoint: eomt_cityscapes.bin) ---


FileNotFoundError: [Errno 2] No such file or directory: '/content/datasets/Validation_Dataset/RoadAnomaly21/images'

MloU Part

In [ ]:
from torchvision.datasets import Cityscapes
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from PIL import Image
import torch, time
import numpy as np

sys.path.insert(0, '/content/drive/MyDrive/Image Segmentation Project/Project/MaskArchitectureAnomaly_CourseProject/eval')
from iouEval import iouEval

CS_ROOT = "/content/drive/MyDrive/Image Segmentation Project/Project/2. Cityscapes"
NUM_CLASSES = 20  # 19 classes + 1 void
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Task 7 (ERFNet) : Resize((512, 1024))
# Task 8 (EoMT)   : Resize((640, 640)) — imposé par le backbone DINOv2
input_transform = Compose([
    ToTensor(),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

cs_val = Cityscapes(CS_ROOT, split='val', mode='fine',
                    target_type='semantic',
                    transform=input_transform)

def collate_fn(batch):
    images = torch.stack([item[0] for item in batch])
    labels = torch.stack([
        torch.from_numpy(np.array(
            dtype=np.int64
        ))
        for item in batch
    ])
    return images, labels

loader = torch.utils.data.DataLoader(cs_val, batch_size=1,
                                      shuffle=False, num_workers=2,
                                      collate_fn=collate_fn)

iou_eval = iouEval(NUM_CLASSES)
start = time.time()

with torch.no_grad():
    for step, (images, labels) in enumerate(loader):
        images = images.to(device)

        labels_train = torch.full_like(labels, 19, dtype=torch.long)
        for cls in Cityscapes.classes:
            if not cls.ignore_in_eval:
                labels_train[labels == cls.id] = cls.train_id
        labels_train = labels_train.to(device)

        outputs = model(images)


        logits = convert_eomt_output_to_pixel_logits(outputs)  # [B, C, H, W]
        # Et dans la boucle :
        logits = F.interpolate(logits, size=(640, 640), mode='bilinear', align_corners=False)

        iou_eval.addBatch(logits.max(1)[1].unsqueeze(1).data,
                          labels_train.unsqueeze(1))

        if step % 100 == 0:
            print(f"[{step}/500] {time.time()-start:.0f}s")

iou_val, _ = iou_eval.getIoU()
print(f"\nEoMT ({CHECKPOINT_NAME}) mIoU: {iou_val*100:.1f}%")

**Temperature Part**

In [ ]:
import torch
import pandas as pd
import numpy as np
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

def evaluate_temperature_scaling(dataset_name, temperatures=[0.5, 0.75, 1.1],
                                  base_dir='/content/datasets/Validation_Dataset'):
    print(f"\n--- Temperature Scaling on {dataset_name} (checkpoint: {CHECKPOINT_NAME}) ---")

    val_dataset = CustomAnomalyDataset(
        root=f'{base_dir}/{dataset_name}',
        input_transform=input_transform_fn,
        target_transform=target_transform_fn
    )
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

    all_gt = []
    # Un buffer par température + MSP de base (t=1.0)
    all_scores = {f't={T}': [] for T in temperatures}
    all_scores['MSP'] = []

    for images, masks, _, _ in tqdm(val_loader, desc=f"Processing"):
        images = images.to(device)
        masks = masks.squeeze(1).numpy()
        h_orig, w_orig = masks.shape[-2], masks.shape[-1]

        outputs = model(images)
        logits = convert_eomt_output_to_pixel_logits(outputs)
        logits = F.interpolate(logits, size=(h_orig, w_orig), mode='bilinear', align_corners=False)

        # MSP de base (t=1.0)
        all_scores['MSP'].append(get_msp_score(logits).astype(np.float16))

        # MSP avec chaque température — même passe, juste on divise les logits
        for T in temperatures:
            logits_t = logits / T
            all_scores[f't={T}'].append(get_msp_score(logits_t).astype(np.float16))

        all_gt.append(masks.astype(np.uint8))

        del images, outputs, logits
        torch.cuda.empty_cache()

    # Calcul des métriques
    gt_concat = np.concatenate(all_gt)
    del all_gt

    results = {}
    for key, scores_list in all_scores.items():
        scores_concat = np.concatenate(scores_list)
        auprc, fpr95 = calculate_anomaly_metrics(gt_concat, scores_concat)
        results[key] = {"AuPRC": round(auprc, 4), "FPR95": round(fpr95, 4)}
        del scores_concat

    del gt_concat
    return results


# Lancement

temperatures = [0.5, 0.75, 1.1]
datasets_to_run = ["RoadAnomaly21", "RoadObsticle21", "FS_LostFound_full", "fs_static", "RoadAnomaly"]

temp_results = {}
with torch.no_grad():
    for d_name in datasets_to_run:
        temp_results[d_name] = evaluate_temperature_scaling(d_name, temperatures)

df_temp = pd.DataFrame(temp_results).round(4)
print("\nRésultats Temperature Scaling :")
print(df_temp.to_string())

# Sauvegarde CSV
csv_path = f'/content/drive/MyDrive/{SHARED_FOLDER_NAME}/Task8_Temperature_Results_{CHECKPOINT_NAME.replace(".bin", "")}.csv'
df_temp.to_csv(csv_path)
print(f"\nSauvegardé : {csv_path}")